In [4]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import pickle

from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("MovieRecEngine_ALS")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

train_pd = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_pd = pd.read_parquet(PROCESSED_DIR / "test.parquet")

train_sdf = spark.createDataFrame(train_pd[["user_idx", "movie_idx", "rating"]])
test_sdf = spark.createDataFrame(test_pd[["user_idx", "movie_idx", "rating"]])

train_sdf = train_sdf.withColumnRenamed("user_idx", "userCol") \
                      .withColumnRenamed("movie_idx", "itemCol") \
                      .withColumn("rating", F.col("rating").cast("float"))

test_sdf = test_sdf.withColumnRenamed("user_idx", "userCol") \
                    .withColumnRenamed("movie_idx", "itemCol") \
                    .withColumn("rating", F.col("rating").cast("float"))

train_sdf.cache()
test_sdf.cache()

evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

final_als = ALS(
    userCol="userCol", itemCol="itemCol", ratingCol="rating",
    rank=10, maxIter=15, regParam=0.05,
    coldStartStrategy="drop", nonnegative=True, seed=42
)
final_model = final_als.fit(train_sdf)

final_preds = final_model.transform(test_sdf)
final_rmse = evaluator_rmse.evaluate(final_preds)
final_mae = evaluator_mae.evaluate(final_preds)

print(f"Final ALS model -> RMSE: {final_rmse:.4f}, MAE: {final_mae:.4f}")

Final ALS model -> RMSE: 0.8750, MAE: 0.6901


In [5]:
user_factors_df = final_model.userFactors.toPandas()
item_factors_df = final_model.itemFactors.toPandas()

print(f"User factors: {user_factors_df.shape}")
print(f"Item factors: {item_factors_df.shape}")
user_factors_df.head()

User factors: (6040, 2)
Item factors: (3667, 2)


,id,features
0,0,"[0.40812567, 0.84494317, 0.85704726, 0.8602605..."
1,10,"[0.0, 0.8410417, 0.0, 0.7993515, 1.4478412, 0...."
2,20,"[0.64853454, 0.17326607, 0.0, 1.5227135, 1.130..."
3,30,"[0.71352214, 0.017437069, 1.0435004, 1.32869, ..."
4,40,"[0.0, 0.0, 0.55255795, 0.0, 1.3911507, 0.0, 0...."
